# Notebook 00 — Experimental Setup

> **Supplementary §1**

This notebook summarises the experimental design, subjects, and data collection protocol for the hawk morphing flight study. It serves as a reference for all subsequent analysis notebooks.

In [1]:
import pandas as pd
import numpy as np
import yaml
import os

# Load the processed frame-level data
frame_info = pd.read_csv("../../data/bilateral/frame_info.csv")

# Hawk name mapping
bird_names = {1: "Drogon", 2: "Rhaegal", 3: "Ruby", 4: "Toothless", 5: "Charmander"}
frame_info["Hawk"] = frame_info["BirdID"].map(bird_names)

print(f"Overview: Loaded {len(frame_info):,} frames from {frame_info['seqID'].nunique():,} flights")

Overview: Loaded 144,764 frames from 1,634 flights


## 1. Experimental Setup

We recorded a total of n = 5 captive-bred Harris' hawks flying between two perches in a purpose-built motion capture flight studio in two experimental periods for **2,067 flights** (total recorded). After filtering for complete frames (all 8 feather markers simultaneously reconstructed), we processed **1,634 flights** yielding **144,764 frames**.

### Experimental Periods

**Period 1 (25 days, N=1,124 processed flights):** Four hawks flew perch-to-perch with different distances (5m, 7m, 9m, 12m) to investigate different braking strategies and capture varying flight behaviours. Three hawks were inexperienced males (<1 year) and one experienced female (7 years).

**Period 2 (4 days, N=510 processed flights):** Experimental setup was replicated for the 9m flights but with the addition of an obstacle to capture asymmetrical manoeuvres, and/or the addition of weight (IMU) onto the hawks.

In [2]:
# Table A: Processed data counts for the two experimental periods and conditions
def condition_label(row):
    if row["Year"] == 2017:
        return f"Period 1 | {int(row['PerchDistance'])}m"
    obs = row["Obstacle"]
    imu = row["IMU"]
    if obs == 0 and imu == 0:
        return "Period 2 | 9m control"
    elif obs == 1 and imu == 0:
        return "Period 2 | 9m obstacle"
    elif obs == 0 and imu == 1:
        return "Period 2 | 9m weight"
    else:
        return "Period 2 | 9m obs+weight"

frame_info["Condition"] = frame_info.apply(condition_label, axis=1)
cond_summary = frame_info.groupby("Condition").agg(
    Flights=("seqID", "nunique"),
    Frames=("Condition", "size")
)

print("**Table A: Processed data counts for conditions**")
print(cond_summary)
print(f"Total processed: {cond_summary['Flights'].sum():,} flights, {cond_summary['Frames'].sum():,} frames")

**Table A: Processed data counts for conditions**
                          Flights  Frames
Condition                                
Period 1 | 12m                327   28674
Period 1 | 5m                 322   17575
Period 1 | 7m                 212   20428
Period 1 | 9m                 263   29647
Period 2 | 9m control         189   22723
Period 2 | 9m obs+weight      127    8009
Period 2 | 9m obstacle        127   10440
Period 2 | 9m weight           67    7268
Total processed: 1,634 flights, 144,764 frames


### Obstacle Turn Breakdown

In obstacle flights, a 2m pole with foam padding was placed at the midpoint (4.5m). Hawks were free to fly either side.

In [3]:
obs_frames = frame_info[frame_info["Obstacle"] == 1]
turn_summary = obs_frames.groupby("Turn").agg(
    Flights=("seqID", "nunique"),
    Frames=("Turn", "size")
)
print("Obstacle Turn Directions:")
for turn, row in turn_summary.iterrows():
    if turn != "Straight":
        print(f"  {turn:5} | {row['Flights']:3} flights | {row['Frames']:6,} frames")

Obstacle Turn Directions:
  Left  | 135 flights |  8,316 frames
  Right | 119 flights | 10,133 frames


In [4]:
# Table B: Processed data counts per individual hawk
hawk_summary = frame_info.groupby(["Hawk", "Naive", "Obstacle", "IMU"]).agg(
    Flights=("seqID", "nunique"),
    Frames=("Hawk", "size")
).reset_index()

print("**Table B: Processed data counts per individual hawk**")
print(f"{'Hawk':<12} | {'Juv':<3} | {'Obs':<3} | {'IMU':<3} | {'Flights':>7} | {'Frames':>8}")
print("-" * 58)
for _, row in hawk_summary.iterrows():
    juv = "✓" if row['Naive'] == 1 else ""
    obs = "✓" if row['Obstacle'] == 1 else ""
    imu = "✓" if row['IMU'] == 1 else ""
    print(f"{row['Hawk']:<12} | {juv:<3} | {obs:<3} | {imu:<3} | {row['Flights']:>7,} | {row['Frames']:>8,}")

**Table B: Processed data counts per individual hawk**
Hawk         | Juv | Obs | IMU | Flights |   Frames
----------------------------------------------------------
Charmander   | ✓   |     |     |      45 |    4,453
Charmander   | ✓   |     | ✓   |      16 |    1,375
Charmander   | ✓   | ✓   |     |      30 |    2,184
Charmander   | ✓   | ✓   | ✓   |      30 |    1,384
Drogon       |     |     |     |      46 |    3,656
Drogon       |     |     | ✓   |      16 |    1,014
Drogon       |     | ✓   |     |      32 |    1,809
Drogon       |     | ✓   | ✓   |      32 |    1,359
Drogon       | ✓   |     |     |     340 |   21,593
Rhaegal      | ✓   |     |     |     327 |   27,900
Ruby         |     |     |     |     161 |   14,273
Ruby         |     |     | ✓   |      18 |    2,197
Ruby         |     | ✓   |     |      32 |    2,499
Ruby         |     | ✓   | ✓   |      32 |    2,383
Toothless    |     |     |     |      50 |    9,046
Toothless    |     |     | ✓   |      17 |    2,682
To

In [5]:
# Table C: Processed data counts per distance (both experimental periods)
control_frames = frame_info[frame_info["Obstacle"] == 0]
table_c = control_frames.groupby(["PerchDistance", "Hawk"]).agg(
    Flights=("seqID", "nunique"),
    Frames=("Hawk", "size")
)

print("**Table C: Processed data counts per distance (non-obstacle flights)**")
print(table_c)

**Table C: Processed data counts per distance (non-obstacle flights)**
                          Flights  Frames
PerchDistance Hawk                       
5             Drogon          149    6862
              Rhaegal          70    4465
              Ruby             41    2811
              Toothless        62    3437
7             Drogon           60    6012
              Rhaegal          72    7090
              Ruby             33    3482
              Toothless        47    3844
9             Charmander       61    5828
              Drogon          138   11945
              Rhaegal         114   14196
              Ruby             81    8552
              Toothless       125   19117
12            Drogon           55    1444
              Rhaegal          71    2149
              Ruby             24    1625
              Toothless       177   23456


In [6]:
# Table D: Estimated wingspans and average measured mass
with open("../../src/kinematic_morphospace/TotalWingspans.yml") as f:
    wingspans = yaml.safe_load(f)

# Mass data from paper records)
masses = {
    "Drogon":    {"2017": 0.65, "2020": 0.66},
    "Rhaegal":   {"2017": 0.62},
    "Ruby":      {"2017": 0.87, "2020": 0.84},
    "Toothless": {"2017": 0.74, "2020": 0.75},
    "Charmander":{"2020": 0.63}
}

print("**Table D: Estimated wingspans and measured mass**")
print(f"{'Name':<12} | {'Year':<4} | {'Wingspan (m)':<12} | {'Mass (kg)':<10}")
print("-" * 50)
for hawk in ["Drogon", "Rhaegal", "Ruby", "Toothless", "Charmander"]:
    if hawk in wingspans:
        for year, ws in wingspans[hawk].items():
            mass = masses.get(hawk, {}).get(str(year), "-")
            print(f"{hawk:<12} | {year:<4} | {ws:<12.4f} | {mass:<10}")

**Table D: Estimated wingspans and measured mass**
Name         | Year | Wingspan (m) | Mass (kg) 
--------------------------------------------------
Drogon       | 2017 | 1.0083       | 0.65      
Drogon       | 2020 | 1.0159       | 0.66      
Rhaegal      | 2017 | 1.0193       | 0.62      
Ruby         | 2017 | 1.0783       | 0.87      
Ruby         | 2020 | 1.0613       | 0.84      
Toothless    | 2017 | 1.0607       | 0.74      
Toothless    | 2020 | 1.0641       | 0.75      
Charmander   | 2020 | 1.0164       | 0.63      


## Figure Frame Counts

Verification of the specific frame and flight counts cited in the main text figure captions.

In [7]:
# === Figure 2: Motion fingerprints ===
# Caption: "Each row shows a different hawk"
print("=== Figure 2: Motion Fingerprints ===\n")

fig2_hawks = {4: "Toothless", 3: "Ruby", 5: "Charmander"}
for bird_id, name in fig2_hawks.items():
    # All flights for this hawk
    mask_all = frame_info["BirdID"] == bird_id
    n_all = mask_all.sum()
    f_all = frame_info.loc[mask_all, "seqID"].nunique()

    # By year
    for year in sorted(frame_info.loc[mask_all, "Year"].unique()):
        mask_yr = mask_all & (frame_info["Year"] == year)
        n_yr = mask_yr.sum()
        f_yr = frame_info.loc[mask_yr, "seqID"].nunique()
        print(f"  {name} {year}: {n_yr:>6,} frames, {f_yr:>3} flights")

    # By year + control only (no obstacle, no IMU)
    for year in sorted(frame_info.loc[mask_all, "Year"].unique()):
        mask_ctrl = mask_all & (frame_info["Year"] == year) & (frame_info["Obstacle"] == 0) & (frame_info["IMU"] == 0)
        n_ctrl = mask_ctrl.sum()
        f_ctrl = frame_info.loc[mask_ctrl, "seqID"].nunique()
        print(f"  {name} {year} (control): {n_ctrl:>6,} frames, {f_ctrl:>3} flights")

    print(f"  {name} TOTAL: {n_all:>6,} frames, {f_all:>3} flights")
    print()

# === Figure 3: Heatmap comparisons ===
print("=== Figure 3: Heatmap Comparisons ===\n")

# Drogon juvenile vs adult
# Juvenile = Naive=1 (Period 1, 2017)
mask_dro_juv = (frame_info["BirdID"] == 1) & (frame_info["Naive"] == 1) & (frame_info["Obstacle"] == 0)
mask_dro_adult = (frame_info["BirdID"] == 1) & (frame_info["Naive"] == 0) & (frame_info["Obstacle"] == 0)
mask_dro_adult_ctrl = mask_dro_adult & (frame_info["IMU"] == 0)

print(f"  Drogon juvenile (naive, no obstacle): "
      f"N={frame_info.loc[mask_dro_juv, 'seqID'].nunique()} flights, "
      f"{mask_dro_juv.sum():,} frames")
print(f"  Drogon adult (experienced, no obstacle): "
      f"N={frame_info.loc[mask_dro_adult, 'seqID'].nunique()} flights, "
      f"{mask_dro_adult.sum():,} frames")
print(f"  Drogon adult (experienced, no obstacle, no IMU): "
      f"N={frame_info.loc[mask_dro_adult_ctrl, 'seqID'].nunique()} flights, "
      f"{mask_dro_adult_ctrl.sum():,} frames")
print()

# Toothless with/without obstacle (2020)
for imu_val, imu_label in [(None, "any IMU"), (0, "no IMU")]:
    for obs_val, obs_label in [(1, "with obstacle"), (0, "without obstacle")]:
        mask = (frame_info["BirdID"] == 4) & (frame_info["Year"] == 2020) & (frame_info["Obstacle"] == obs_val)
        if imu_val is not None:
            mask = mask & (frame_info["IMU"] == imu_val)
        n_flights = frame_info.loc[mask, "seqID"].nunique()
        print(f"  Toothless 2020 {obs_label} ({imu_label}): "
              f"N={n_flights} flights, {mask.sum():,} frames")
print()

# Ruby with/without weight (2020)
for obs_val, obs_label in [(None, "any obstacle"), (0, "no obstacle")]:
    for imu_val, imu_label in [(1, "with weight"), (0, "without weight")]:
        mask = (frame_info["BirdID"] == 3) & (frame_info["Year"] == 2020) & (frame_info["IMU"] == imu_val)
        if obs_val is not None:
            mask = mask & (frame_info["Obstacle"] == obs_val)
        n_flights = frame_info.loc[mask, "seqID"].nunique()
        print(f"  Ruby 2020 {imu_label} ({obs_label}): "
              f"N={n_flights} flights, {mask.sum():,} frames")

=== Figure 2: Motion Fingerprints ===

  Toothless 2017: 38,126 frames, 344 flights
  Toothless 2020: 18,559 frames, 133 flights
  Toothless 2017 (control): 38,126 frames, 344 flights
  Toothless 2020 (control):  9,046 frames,  50 flights
  Toothless TOTAL: 56,685 frames, 477 flights

  Ruby 2017:  8,705 frames, 113 flights
  Ruby 2020: 12,647 frames, 130 flights
  Ruby 2017 (control):  8,705 frames, 113 flights
  Ruby 2020 (control):  5,568 frames,  48 flights
  Ruby TOTAL: 21,352 frames, 243 flights

  Charmander 2020:  9,396 frames, 121 flights
  Charmander 2020 (control):  4,453 frames,  45 flights
  Charmander TOTAL:  9,396 frames, 121 flights

=== Figure 3: Heatmap Comparisons ===

  Drogon juvenile (naive, no obstacle): N=340 flights, 21,593 frames
  Drogon adult (experienced, no obstacle): N=62 flights, 4,670 frames
  Drogon adult (experienced, no obstacle, no IMU): N=46 flights, 3,656 frames

  Toothless 2020 with obstacle (any IMU): N=66 flights, 6,831 frames
  Toothless 2020

## Marker Setup

Each hawk wore:
- **Backpack:** Rigid marker template (6.4 mm rigid body markers) near the centre of mass (origin).
- **Tail-clip:** Rigid marker template (6.4 mm rigid body markers) on central tail feathers (body vector).
- **Eight feather markers:** 2 mm hemispherical markers on left/right Wingtip, Primary, Secondary, and Tail-tip.

All markers were attached using double-sided tape. The choice of marker size and attachment method was constrained by bird welfare protocols: markers had to be lightweight (minimising flight impairment), non-invasive (no adhesive on skin), and easily removable after each session. The 2 mm hemispherical feather markers were the smallest that could be reliably tracked by the Vicon system at the required capture volume.

## 2. Marker Reconstruction

Tracking with 20-camera Vicon Vantage system. Coordinate system aligned with flight hall. Flights with >30% missing backpack data or missed landings removed.

### 2.1 Trajectories

Backpack and tail-pack used for body vector. Origin at destination perch. Frames before take-off (<-0.5s) and after landing (<0.1m) removed. 

In [8]:
# Trajectory verification (2.1, 2.2)
# Compute counts from raw data rather than hardcoding
import contextlib, io
from kinematic_morphospace import load_data, remove_frames

traj_raw = load_data("../../data/raw/trajectories.csv")
n_starting = len(traj_raw)

# Suppress verbose remove_frames output
with contextlib.redirect_stdout(io.StringIO()):
    traj_filtered = remove_frames(traj_raw, Y_limit=0.1, time_limit=-0.5)
n_filtered = len(traj_filtered)

print(f"Starting frames with backpack (initial count): {n_starting:,}")
print(f"Processed frames after filtering: {n_filtered:,}")
print(f"Final 8-marker complete frames used in study: {len(frame_info):,}")

Starting frames with backpack (initial count): 939,383
Processed frames after filtering: 814,096
Final 8-marker complete frames used in study: 144,764


## 3. Labelling Feather Markers

Relative distances from feather markers to backpack used for labelling. 559,421 labelled frames total from 2,067 flights. Only **144,764** frames had all eight markers simultaneously labelled.

## 4. Principal Component Analysis

PCA identifies linear combinations capturing maximum variance. Used to reveal coupled wing-tail motions.

### 4.1 Scaling and Normalisation

Marker positions relative to centre of mass and scaled to each hawk's maximum wingspan. **Z-score normalisation is NOT used** as it would distort spatial relationships between markers.

## Notebook Processing Sequence

1. **01 — Trajectories:** Load trajectory data and whole-body kinematics.
2. **02 — Bilateral PCA:** PCA on full bilateral data.
3. **03 — Rotation Correction:** Symmetry-based rotation removal.
4. **04 — Morphing Shape Modes:** Unilateral morphing analysis.

---

## References

- KleinHeerenbrink, M., France, L. A., Brighton, C. H. & Taylor, G. K. (2022). Optimization of avian perching manoeuvres. *Nature*, 607, 91–96.